# Weighted Combiner with GO DAG Propagation

This notebook builds an integration model for the CAFA team predictions using:

- Sequence team predictions
- Structure team predictions
- PROTGOAT predictions

It does the following for each ontology (`BPO`, `CCO`, `MFO`):

1. Loads and standardizes predictions
2. Builds the union of predicted `(protein_id, go_term)` pairs
3. Merges the three model scores
4. Tunes a simple weighted average on the training set
5. Applies GO DAG upward propagation
6. Tunes a threshold
7. Writes CAFA-style train and test outputs

The path setup below matches the local folder structure you showed on your computer.


In [1]:

from pathlib import Path
import itertools
import math
import re
import zipfile
from collections import defaultdict, deque

import numpy as np
import pandas as pd


## 1. Local paths

This cell is set up for your current folder layout:

- `CAFA Training Data`
- `PROTGOAT Predictions`
- `Sequence Team Predictions`
- `Structure Team Predictions`

all sitting next to the notebook.


In [2]:
from pathlib import Path
import os

print("cwd:", Path.cwd())
print("cwd exists:", Path.cwd().exists())
print("contents of cwd:")
for p in Path.cwd().iterdir():
    print(" -", p.name)

cwd: /Users/erikaholbrook/CS 167/CS 167 Final Project CAFA5
cwd exists: True
contents of cwd:
 - CAFA5 General Data 
 - integration_outputs
 - .ipynb_checkpoints
 - Integration Team Working Folder
 - PROTGOAT Model


In [3]:
from pathlib import Path
import numpy as np

# We are currently one level ABOVE the working folder
BASE_DIR = Path.cwd() / "Integration Team Working Folder"

print("BASE_DIR:", BASE_DIR)
print("BASE_DIR exists:", BASE_DIR.exists())

CAFA_DIR = BASE_DIR / "CAFA Training Data"
SEQ_DIR = BASE_DIR / "Sequence Team Predictions"
STRUCT_DIR = BASE_DIR / "Structure Team Predictions"
PROTGOAT_DIR = BASE_DIR / "PROTGOAT Predictions"

OUTPUT_DIR = BASE_DIR / "integration_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# CAFA files
TRAIN_TERMS_PATH = CAFA_DIR / "train_terms.tsv"
TRAIN_TAXONOMY_PATH = CAFA_DIR / "train_taxonomy.tsv"
GO_OBO_PATH = CAFA_DIR / "go-basic.obo"
IA_PATH = CAFA_DIR / "IA(3).txt"
SAMPLE_SUBMISSION_PATH = CAFA_DIR / "sample_submission.tsv"

# Sequence team files
SEQ_TRAIN_BP_PATH = SEQ_DIR / "raw_train_sequences_preds_bp_trim.tsv"
SEQ_TRAIN_CC_PATH = SEQ_DIR / "raw_train_sequences_preds_cc_trim.tsv"
SEQ_TRAIN_MF_PATH = SEQ_DIR / "raw_train_sequences_preds_mf_trim.tsv"

SEQ_TEST_BP_PATH = SEQ_DIR / "testsuperset_unique_preds_bp_trim.tsv"
SEQ_TEST_CC_PATH = SEQ_DIR / "testsuperset_unique_preds_cc_trim.tsv"
SEQ_TEST_MF_PATH = SEQ_DIR / "testsuperset_unique_preds_mf_trim.tsv"

# Structure team files
STRUCT_TRAIN_PATH = STRUCT_DIR / "structure_training_predictions_combined.tsv"
STRUCT_TEST_PATH = STRUCT_DIR / "structure_test_predictions_combined.tsv"

# PROTGOAT files
PROTGOAT_TRAIN_PATH = PROTGOAT_DIR / "protgoat_train_predictions.tsv"
PROTGOAT_TEST_PATH = PROTGOAT_DIR / "protgoat_test_predictions.tsv"

# Outputs
TRAIN_OUTPUT_PATH = OUTPUT_DIR / "integration_train_predictions.tsv"
TEST_OUTPUT_PATH = OUTPUT_DIR / "integration_test_predictions.tsv"
TUNING_SUMMARY_PATH = OUTPUT_DIR / "tuning_summary.tsv"

WEIGHT_STEP = 0.10
THRESHOLDS = np.round(np.arange(0.05, 1.00, 0.05), 2)
THRESHOLD_TRAIN_OUTPUT = True


BASE_DIR: /Users/erikaholbrook/CS 167/CS 167 Final Project CAFA5/Integration Team Working Folder
BASE_DIR exists: True


In [4]:
print("BASE_DIR contents:")
for p in BASE_DIR.iterdir():
    print("-", repr(p.name))

BASE_DIR contents:
- 'PROTGOAT Predictions'
- 'integration_outputs'
- 'CAFA Training Data'
- '.ipynb_checkpoints'
- 'Structure Team Predictions'
- 'Sequence Team Predictions'
- 'weighted_combiner_dag_integration_notebook_local_paths.ipynb'


In [5]:
from pathlib import Path

print("cwd:", Path.cwd())
print("\ncwd contents:")
for p in Path.cwd().iterdir():
    print("-", repr(p.name))

parent = Path.cwd().parent
print("\nparent:", parent)
print("\nparent contents:")
for p in parent.iterdir():
    print("-", repr(p.name))

cwd: /Users/erikaholbrook/CS 167/CS 167 Final Project CAFA5

cwd contents:
- 'CAFA5 General Data '
- 'integration_outputs'
- '.ipynb_checkpoints'
- 'Integration Team Working Folder'
- 'PROTGOAT Model'

parent: /Users/erikaholbrook/CS 167

parent contents:
- '.DS_Store'
- 'CS 167 HW2'
- 'CS 167 HW3'
- 'CS 167 Final Project CAFA5'
- '.ipynb_checkpoints'


In [6]:
print("BASE_DIR:", BASE_DIR)
print("CAFA_DIR exists:", CAFA_DIR.exists())
print("SEQ_DIR exists:", SEQ_DIR.exists())
print("STRUCT_DIR exists:", STRUCT_DIR.exists())
print("PROTGOAT_DIR exists:", PROTGOAT_DIR.exists())

print("TRAIN_TERMS_PATH exists:", TRAIN_TERMS_PATH.exists())
print("TRAIN_TAXONOMY_PATH exists:", TRAIN_TAXONOMY_PATH.exists())
print("GO_OBO_PATH exists:", GO_OBO_PATH.exists())
print("IA_PATH exists:", IA_PATH.exists())
print("SAMPLE_SUBMISSION_PATH exists:", SAMPLE_SUBMISSION_PATH.exists())

print("SEQ_TRAIN_BP_PATH exists:", SEQ_TRAIN_BP_PATH.exists())
print("SEQ_TRAIN_CC_PATH exists:", SEQ_TRAIN_CC_PATH.exists())
print("SEQ_TRAIN_MF_PATH exists:", SEQ_TRAIN_MF_PATH.exists())

print("SEQ_TEST_BP_PATH exists:", SEQ_TEST_BP_PATH.exists())
print("SEQ_TEST_CC_PATH exists:", SEQ_TEST_CC_PATH.exists())
print("SEQ_TEST_MF_PATH exists:", SEQ_TEST_MF_PATH.exists())

print("STRUCT_TRAIN_PATH exists:", STRUCT_TRAIN_PATH.exists())
print("STRUCT_TEST_PATH exists:", STRUCT_TEST_PATH.exists())

print("PROTGOAT_TRAIN_PATH exists:", PROTGOAT_TRAIN_PATH.exists())
print("PROTGOAT_TEST_PATH exists:", PROTGOAT_TEST_PATH.exists())

BASE_DIR: /Users/erikaholbrook/CS 167/CS 167 Final Project CAFA5/Integration Team Working Folder
CAFA_DIR exists: True
SEQ_DIR exists: True
STRUCT_DIR exists: True
PROTGOAT_DIR exists: True
TRAIN_TERMS_PATH exists: True
TRAIN_TAXONOMY_PATH exists: True
GO_OBO_PATH exists: True
IA_PATH exists: False
SAMPLE_SUBMISSION_PATH exists: True
SEQ_TRAIN_BP_PATH exists: True
SEQ_TRAIN_CC_PATH exists: True
SEQ_TRAIN_MF_PATH exists: True
SEQ_TEST_BP_PATH exists: True
SEQ_TEST_CC_PATH exists: True
SEQ_TEST_MF_PATH exists: True
STRUCT_TRAIN_PATH exists: True
STRUCT_TEST_PATH exists: True
PROTGOAT_TRAIN_PATH exists: True
PROTGOAT_TEST_PATH exists: True


In [7]:

print("BASE_DIR:", BASE_DIR)
print("CAFA_DIR exists:", CAFA_DIR.exists())
print("SEQ_DIR exists:", SEQ_DIR.exists())
print("STRUCT_DIR exists:", STRUCT_DIR.exists())
print("PROTGOAT_DIR exists:", PROTGOAT_DIR.exists())


BASE_DIR: /Users/erikaholbrook/CS 167/CS 167 Final Project CAFA5/Integration Team Working Folder
CAFA_DIR exists: True
SEQ_DIR exists: True
STRUCT_DIR exists: True
PROTGOAT_DIR exists: True


## 2. Optional unzip helper

Run this only if the structure or PROTGOAT TSV files are still zipped in their folders.
If you already unzipped them, you can skip this cell.


In [8]:

def unzip_if_needed(zip_path, extract_dir):
    if zip_path.exists():
        print(f"Unzipping {zip_path.name} -> {extract_dir}")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_dir)

# Uncomment these if needed:
# unzip_if_needed(STRUCT_DIR / "structure_training_predictions_combined.tsv.zip", STRUCT_DIR)
# unzip_if_needed(STRUCT_DIR / "structure_test_predictions_combined.tsv.zip", STRUCT_DIR)
# unzip_if_needed(PROTGOAT_DIR / "protgoat_train_predictions.tsv.zip", PROTGOAT_DIR)
# unzip_if_needed(PROTGOAT_DIR / "protgoat_test_predictions.tsv.zip", PROTGOAT_DIR)


## 3. File readers

In [9]:

def read_sequence_train(path):
    """
    Sequence train files came in whitespace-separated without headers.
    """
    df = pd.read_csv(
        path,
        sep=r"\s+",
        header=None,
        names=["protein_id", "go_term", "confidence"],
        engine="python"
    )
    return df

def read_sequence_test(path):
    """
    Sequence test files came in tab-separated without headers.
    """
    df = pd.read_csv(
        path,
        sep="\t",
        header=None,
        names=["protein_id", "go_term", "confidence"]
    )
    return df

def read_standard_prediction_tsv(path):
    """
    Structure and PROTGOAT files are tab-separated without headers.
    """
    df = pd.read_csv(
        path,
        sep="\t",
        header=None,
        names=["protein_id", "go_term", "confidence"]
    )
    return df

def clean_prediction_df(df, source_name):
    df = df.copy()
    df["protein_id"] = df["protein_id"].astype(str).str.strip()
    df["go_term"] = df["go_term"].astype(str).str.strip()
    df["confidence"] = pd.to_numeric(df["confidence"], errors="coerce").fillna(0.0)
    df = df.dropna(subset=["protein_id", "go_term"])
    df = df[(df["protein_id"] != "") & (df["go_term"] != "")]
    df = df.groupby(["protein_id", "go_term"], as_index=False)["confidence"].max()
    df["source"] = source_name
    return df


## 4. Load CAFA truth data

In [10]:

train_terms = pd.read_csv(TRAIN_TERMS_PATH, sep="\t")
train_terms = train_terms.rename(columns={"EntryID": "protein_id", "term": "go_term", "aspect": "aspect"})
train_terms["protein_id"] = train_terms["protein_id"].astype(str).str.strip()
train_terms["go_term"] = train_terms["go_term"].astype(str).str.strip()
train_terms["aspect"] = train_terms["aspect"].astype(str).str.strip()

train_terms.head()


,protein_id,go_term,aspect
0,A0A009IHW8,GO:0008152,BPO
1,A0A009IHW8,GO:0034655,BPO
2,A0A009IHW8,GO:0072523,BPO
3,A0A009IHW8,GO:0044270,BPO
4,A0A009IHW8,GO:0006753,BPO


In [11]:

train_taxonomy = pd.read_csv(TRAIN_TAXONOMY_PATH, sep="\t")
train_taxonomy = train_taxonomy.rename(columns={"EntryID": "protein_id", "taxonomyID": "taxonomy_id"})
train_taxonomy["protein_id"] = train_taxonomy["protein_id"].astype(str).str.strip()

train_taxonomy.head()


,protein_id,taxonomy_id
0,Q8IXT2,9606
1,Q04418,559292
2,A8DYA3,7227
3,Q9UUI3,284812
4,Q57ZS4,185431


## 5. Parse the GO DAG from `go-basic.obo`

We need:
- term namespace (`BPO`, `CCO`, `MFO`)
- parent relationships
- ancestor lookup for upward propagation


In [11]:

def parse_go_obo(obo_path):
    go_namespace = {}
    parents = defaultdict(set)
    children = defaultdict(set)

    current_id = None
    current_namespace = None
    current_parents = []

    with open(obo_path, "r", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.strip()

            if line == "[Term]":
                if current_id is not None:
                    go_namespace[current_id] = current_namespace
                    for p in current_parents:
                        parents[current_id].add(p)
                        children[p].add(current_id)

                current_id = None
                current_namespace = None
                current_parents = []
                continue

            if line.startswith("id: GO:"):
                current_id = line.replace("id: ", "").strip()
            elif line.startswith("namespace: "):
                ns = line.replace("namespace: ", "").strip()
                if ns == "biological_process":
                    current_namespace = "BPO"
                elif ns == "cellular_component":
                    current_namespace = "CCO"
                elif ns == "molecular_function":
                    current_namespace = "MFO"
            elif line.startswith("is_a: GO:"):
                parent_id = line.split()[1]
                current_parents.append(parent_id)

        if current_id is not None:
            go_namespace[current_id] = current_namespace
            for p in current_parents:
                parents[current_id].add(p)
                children[p].add(current_id)

    return go_namespace, parents, children

go_namespace, go_parents, go_children = parse_go_obo(GO_OBO_PATH)

print("GO terms with namespace:", len(go_namespace))
print("Terms with parents:", sum(len(v) > 0 for v in go_parents.values()))


GO terms with namespace: 47417
Terms with parents: 43245


In [12]:

def build_ancestor_lookup(go_namespace, go_parents):
    ancestor_lookup = {}

    for term in go_namespace.keys():
        visited = set()
        stack = list(go_parents.get(term, []))

        while stack:
            node = stack.pop()
            if node in visited:
                continue
            visited.add(node)
            stack.extend(go_parents.get(node, []))

        ancestor_lookup[term] = visited

    return ancestor_lookup

ancestor_lookup = build_ancestor_lookup(go_namespace, go_parents)
print("Ancestor lookup built.")


Ancestor lookup built.


## 6. Load all model predictions

In [13]:

# Sequence
seq_train_bp = clean_prediction_df(read_sequence_train(SEQ_TRAIN_BP_PATH), "sequence")
seq_train_cc = clean_prediction_df(read_sequence_train(SEQ_TRAIN_CC_PATH), "sequence")
seq_train_mf = clean_prediction_df(read_sequence_train(SEQ_TRAIN_MF_PATH), "sequence")

seq_test_bp = clean_prediction_df(read_sequence_test(SEQ_TEST_BP_PATH), "sequence")
seq_test_cc = clean_prediction_df(read_sequence_test(SEQ_TEST_CC_PATH), "sequence")
seq_test_mf = clean_prediction_df(read_sequence_test(SEQ_TEST_MF_PATH), "sequence")

seq_train_bp["aspect"] = "BPO"
seq_train_cc["aspect"] = "CCO"
seq_train_mf["aspect"] = "MFO"

seq_test_bp["aspect"] = "BPO"
seq_test_cc["aspect"] = "CCO"
seq_test_mf["aspect"] = "MFO"

seq_train_all = pd.concat([seq_train_bp, seq_train_cc, seq_train_mf], ignore_index=True)
seq_test_all = pd.concat([seq_test_bp, seq_test_cc, seq_test_mf], ignore_index=True)

# Structure
struct_train_all = clean_prediction_df(read_standard_prediction_tsv(STRUCT_TRAIN_PATH), "structure")
struct_test_all = clean_prediction_df(read_standard_prediction_tsv(STRUCT_TEST_PATH), "structure")

# PROTGOAT
protgoat_train_all = clean_prediction_df(read_standard_prediction_tsv(PROTGOAT_TRAIN_PATH), "protgoat")
protgoat_test_all = clean_prediction_df(read_standard_prediction_tsv(PROTGOAT_TEST_PATH), "protgoat")

# Add ontology labels to combined files using GO namespace
struct_train_all["aspect"] = struct_train_all["go_term"].map(go_namespace)
struct_test_all["aspect"] = struct_test_all["go_term"].map(go_namespace)
protgoat_train_all["aspect"] = protgoat_train_all["go_term"].map(go_namespace)
protgoat_test_all["aspect"] = protgoat_test_all["go_term"].map(go_namespace)

# Keep only valid ontology-mapped terms
struct_train_all = struct_train_all[struct_train_all["aspect"].isin(["BPO", "CCO", "MFO"])].copy()
struct_test_all = struct_test_all[struct_test_all["aspect"].isin(["BPO", "CCO", "MFO"])].copy()
protgoat_train_all = protgoat_train_all[protgoat_train_all["aspect"].isin(["BPO", "CCO", "MFO"])].copy()
protgoat_test_all = protgoat_test_all[protgoat_test_all["aspect"].isin(["BPO", "CCO", "MFO"])].copy()

print("Sequence train rows:", len(seq_train_all))
print("Structure train rows:", len(struct_train_all))
print("PROTGOAT train rows:", len(protgoat_train_all))


Sequence train rows: 21059350
Structure train rows: 11953066
PROTGOAT train rows: 7782584


In [14]:

seq_train_all.head()


,protein_id,go_term,confidence,source,aspect
0,A0A009IHW8,GO:0006139,0.549,sequence,BPO
1,A0A009IHW8,GO:0006259,0.154,sequence,BPO
2,A0A009IHW8,GO:0006351,0.257,sequence,BPO
3,A0A009IHW8,GO:0006355,0.215,sequence,BPO
4,A0A009IHW8,GO:0006508,0.173,sequence,BPO


## 7. Helper functions for merging, scoring, and evaluation

In [15]:

def make_union_table(pred_dfs, aspect):
    """
    Builds the union of (protein_id, go_term) pairs across all sources for one ontology.
    """
    parts = []
    for df in pred_dfs:
        tmp = df[df["aspect"] == aspect][["protein_id", "go_term"]].drop_duplicates()
        parts.append(tmp)

    union_df = pd.concat(parts, ignore_index=True).drop_duplicates()
    return union_df

def attach_confidence(base_df, pred_df, new_col, aspect):
    tmp = pred_df[pred_df["aspect"] == aspect][["protein_id", "go_term", "confidence"]].copy()
    tmp = tmp.rename(columns={"confidence": new_col})
    out = base_df.merge(tmp, on=["protein_id", "go_term"], how="left")
    out[new_col] = out[new_col].fillna(0.0)
    return out

def build_train_table(aspect, seq_df, struct_df, protgoat_df, truth_df):
    base = make_union_table([seq_df, struct_df, protgoat_df], aspect)

    base = attach_confidence(base, seq_df, "conf_seq", aspect)
    base = attach_confidence(base, struct_df, "conf_struct", aspect)
    base = attach_confidence(base, protgoat_df, "conf_protgoat", aspect)

    truth_pairs = truth_df[truth_df["aspect"] == aspect][["protein_id", "go_term"]].drop_duplicates().copy()
    truth_pairs["label"] = 1

    base = base.merge(truth_pairs, on=["protein_id", "go_term"], how="left")
    base["label"] = base["label"].fillna(0).astype(int)
    base["aspect"] = aspect
    return base

def build_test_table(aspect, seq_df, struct_df, protgoat_df):
    base = make_union_table([seq_df, struct_df, protgoat_df], aspect)

    base = attach_confidence(base, seq_df, "conf_seq", aspect)
    base = attach_confidence(base, struct_df, "conf_struct", aspect)
    base = attach_confidence(base, protgoat_df, "conf_protgoat", aspect)

    base["aspect"] = aspect
    return base

def weighted_score(df, weights):
    w_seq, w_struct, w_protgoat = weights
    return (
        w_seq * df["conf_seq"].to_numpy() +
        w_struct * df["conf_struct"].to_numpy() +
        w_protgoat * df["conf_protgoat"].to_numpy()
    )

def precision_recall_f1(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    return precision, recall, f1

def generate_weight_grid(step=0.10):
    values = np.round(np.arange(0.0, 1.0 + step, step), 10)
    grid = []
    for a in values:
        for b in values:
            c = round(1.0 - a - b, 10)
            if c < -1e-9:
                continue
            if c < 0:
                c = 0.0
            if abs((a + b + c) - 1.0) < 1e-8:
                grid.append((round(a, 10), round(b, 10), round(c, 10)))

    grid = sorted(set(grid))
    return grid


## 8. Upward DAG propagation

If a child term has score `x`, then each ancestor should end up with score at least `x`.


In [16]:

def propagate_scores_upward(df, ancestor_lookup):
    """
    Input:
        df with columns [protein_id, go_term, confidence]
    Output:
        propagated df with max score kept per (protein_id, go_term)
    """
    protein_to_scores = defaultdict(dict)

    for row in df.itertuples(index=False):
        protein = row.protein_id
        term = row.go_term
        score = float(row.confidence)

        current = protein_to_scores[protein].get(term, 0.0)
        if score > current:
            protein_to_scores[protein][term] = score

        for anc in ancestor_lookup.get(term, set()):
            current_anc = protein_to_scores[protein].get(anc, 0.0)
            if score > current_anc:
                protein_to_scores[protein][anc] = score

    rows = []
    for protein, score_map in protein_to_scores.items():
        for term, score in score_map.items():
            rows.append((protein, term, score))

    out = pd.DataFrame(rows, columns=["protein_id", "go_term", "confidence"])
    return out


## 9. Tune weights and thresholds by ontology

This uses a simple in-sample search on the train candidate table.
It is not a fancy CV setup, but it gives you a reasonable first integration model.


In [17]:

def tune_weights_and_threshold(train_df, step=0.10, thresholds=None):
    if thresholds is None:
        thresholds = np.round(np.arange(0.05, 1.00, 0.05), 2)

    weight_grid = generate_weight_grid(step)
    best = {
        "weights": None,
        "threshold": None,
        "precision": None,
        "recall": None,
        "f1": -1.0
    }

    y_true = train_df["label"].to_numpy()

    for weights in weight_grid:
        raw_scores = weighted_score(train_df, weights)

        for thr in thresholds:
            y_pred = (raw_scores >= thr).astype(int)
            precision, recall, f1 = precision_recall_f1(y_true, y_pred)

            if f1 > best["f1"]:
                best = {
                    "weights": weights,
                    "threshold": float(thr),
                    "precision": float(precision),
                    "recall": float(recall),
                    "f1": float(f1)
                }

    return best


## 10. Build ontology-specific train/test tables

In [18]:

train_tables = {}
test_tables = {}

for aspect in ["BPO", "CCO", "MFO"]:
    train_tables[aspect] = build_train_table(
        aspect,
        seq_train_all,
        struct_train_all,
        protgoat_train_all,
        train_terms
    )
    test_tables[aspect] = build_test_table(
        aspect,
        seq_test_all,
        struct_test_all,
        protgoat_test_all
    )

for aspect in ["BPO", "CCO", "MFO"]:
    print(aspect, "train rows:", len(train_tables[aspect]), "test rows:", len(test_tables[aspect]))


BPO train rows: 20623723 test rows: 10984302
CCO train rows: 5040331 test rows: 2978519
MFO train rows: 3389373 test rows: 1787336


In [19]:

train_tables["BPO"].head()


,protein_id,go_term,conf_seq,conf_struct,conf_protgoat,label,aspect
0,A0A009IHW8,GO:0006139,0.549,0.359,0.000,1,BPO
1,A0A009IHW8,GO:0006259,0.154,0.113,0.167,0,BPO
2,A0A009IHW8,GO:0006351,0.257,0.000,0.000,0,BPO
3,A0A009IHW8,GO:0006355,0.215,0.000,0.000,0,BPO
4,A0A009IHW8,GO:0006508,0.173,0.000,0.000,0,BPO


## 11. Tune each ontology

In [20]:

tuning_rows = []

for aspect in ["BPO", "CCO", "MFO"]:
    print(f"Tuning {aspect}...")
    best = tune_weights_and_threshold(
        train_tables[aspect],
        step=WEIGHT_STEP,
        thresholds=THRESHOLDS
    )

    w_seq, w_struct, w_protgoat = best["weights"]

    tuning_rows.append({
        "aspect": aspect,
        "w_seq": w_seq,
        "w_struct": w_struct,
        "w_protgoat": w_protgoat,
        "threshold": best["threshold"],
        "precision": best["precision"],
        "recall": best["recall"],
        "f1": best["f1"]
    })

tuning_summary = pd.DataFrame(tuning_rows).sort_values("aspect").reset_index(drop=True)
tuning_summary


Tuning BPO...
Tuning CCO...
Tuning MFO...


,aspect,w_seq,w_struct,w_protgoat,threshold,precision,recall,f1
0,BPO,0.0,0.3,0.7,0.30,0.650072,0.661893,0.655929
1,CCO,0.0,0.4,0.6,0.35,0.891568,0.829951,0.859657
2,MFO,0.0,0.3,0.7,0.30,0.872045,0.811566,0.840719


In [21]:

tuning_summary.to_csv(TUNING_SUMMARY_PATH, sep="\t", index=False)
print("Saved:", TUNING_SUMMARY_PATH)


Saved: /Users/erikaholbrook/CS 167/CS 167 Final Project CAFA5/Integration Team Working Folder/integration_outputs/tuning_summary.tsv


## 12. Score train and test, then propagate up the GO DAG

In [22]:

def score_table(df, weights):
    out = df.copy()
    out["confidence"] = weighted_score(out, weights)
    return out[["protein_id", "go_term", "aspect", "confidence"]]

train_scored_list = []
test_scored_list = []

for row in tuning_summary.itertuples(index=False):
    aspect = row.aspect
    weights = (row.w_seq, row.w_struct, row.w_protgoat)

    train_scored = score_table(train_tables[aspect], weights)
    test_scored = score_table(test_tables[aspect], weights)

    train_scored_list.append(train_scored)
    test_scored_list.append(test_scored)

train_scored_all = pd.concat(train_scored_list, ignore_index=True)
test_scored_all = pd.concat(test_scored_list, ignore_index=True)

train_scored_all.head()


,protein_id,go_term,aspect,confidence
0,A0A009IHW8,GO:0006139,BPO,0.1077
1,A0A009IHW8,GO:0006259,BPO,0.1508
2,A0A009IHW8,GO:0006351,BPO,0.0000
3,A0A009IHW8,GO:0006355,BPO,0.0000
4,A0A009IHW8,GO:0006508,BPO,0.0000


In [23]:

# Propagate ontology by ontology so we do not mix namespaces.
propagated_train_parts = []
propagated_test_parts = []

for aspect in ["BPO", "CCO", "MFO"]:
    train_piece = train_scored_all[train_scored_all["aspect"] == aspect][["protein_id", "go_term", "confidence"]].copy()
    test_piece = test_scored_all[test_scored_all["aspect"] == aspect][["protein_id", "go_term", "confidence"]].copy()

    train_prop = propagate_scores_upward(train_piece, ancestor_lookup)
    test_prop = propagate_scores_upward(test_piece, ancestor_lookup)

    train_prop["aspect"] = train_prop["go_term"].map(go_namespace)
    test_prop["aspect"] = test_prop["go_term"].map(go_namespace)

    train_prop = train_prop[train_prop["aspect"] == aspect].copy()
    test_prop = test_prop[test_prop["aspect"] == aspect].copy()

    propagated_train_parts.append(train_prop)
    propagated_test_parts.append(test_prop)

propagated_train_all = pd.concat(propagated_train_parts, ignore_index=True)
propagated_test_all = pd.concat(propagated_test_parts, ignore_index=True)

print("Propagated train rows:", len(propagated_train_all))
print("Propagated test rows:", len(propagated_test_all))


Propagated train rows: 17157218
Propagated test rows: 16080052


## 13. Apply ontology-specific thresholds

In [ ]:

threshold_map = dict(zip(tuning_summary["aspect"], tuning_summary["threshold"]))

if THRESHOLD_TRAIN_OUTPUT:
    final_train = propagated_train_all[
        propagated_train_all.apply(lambda r: r["confidence"] >= threshold_map.get(r["aspect"], 1.0), axis=1)
    ].copy()
else:
    final_train = propagated_train_all.copy()

final_test = propagated_test_all[
    propagated_test_all.apply(lambda r: r["confidence"] >= threshold_map.get(r["aspect"], 1.0), axis=1)
].copy()

# Keep max confidence per (protein, term)
final_train = final_train.groupby(["protein_id", "go_term"], as_index=False)["confidence"].max()
final_test = final_test.groupby(["protein_id", "go_term"], as_index=False)["confidence"].max()

# Sort for cleaner output
final_train = final_train.sort_values(["protein_id", "go_term"]).reset_index(drop=True)
final_test = final_test.sort_values(["protein_id", "go_term"]).reset_index(drop=True)

print("Final train rows:", len(final_train))
print("Final test rows:", len(final_test))


## 14. Save CAFA-style outputs

In [ ]:

final_train.to_csv(TRAIN_OUTPUT_PATH, sep="\t", index=False, header=False)
final_test.to_csv(TEST_OUTPUT_PATH, sep="\t", index=False, header=False)

print("Saved train output:", TRAIN_OUTPUT_PATH)
print("Saved test output:", TEST_OUTPUT_PATH)
print("Saved tuning summary:", TUNING_SUMMARY_PATH)


In [ ]:

print("Tuning summary")
display(tuning_summary)

print("\nTrain preview")
display(final_train.head(10))

print("\nTest preview")
display(final_test.head(10))


## 15. Optional quick checks

These are useful if you want to sanity-check what the notebook produced.


In [ ]:

# How many predictions per ontology after thresholding?
train_with_aspect = final_train.copy()
test_with_aspect = final_test.copy()

train_with_aspect["aspect"] = train_with_aspect["go_term"].map(go_namespace)
test_with_aspect["aspect"] = test_with_aspect["go_term"].map(go_namespace)

print("Train counts by aspect")
display(train_with_aspect["aspect"].value_counts(dropna=False))

print("Test counts by aspect")
display(test_with_aspect["aspect"].value_counts(dropna=False))


## Notes

- If the notebook says a file is missing, check the path cell first.
- If Jupyter opens from the wrong working directory, replace `BASE_DIR = Path.cwd()` with your exact folder path.
- If the weight search feels slow, start with `WEIGHT_STEP = 0.20`.
- If you want a finer search later, try `WEIGHT_STEP = 0.05`.
